# Wildfire Persistence and Spread in the Pantanal — hourly forcing

Hourly counterpart of the laboratory's daily notebook (`ml_persist_propaga_wind.ipynb`,
not included in this repository), restricted to the
**Pantanal** and covering **2019–2024**, using `hourly_ml_table_pantanal.parquet`
(built by `build_hourly_dataset.py`).

### What "hourly" actually means here

The fire product is a **single afternoon overpass**: Pantanal detections land
only between 16 and 19 UTC (mostly 17 and 18), and the overwhelming majority of fire days carry
exactly one distinct detection hour. There is no observation at `t+1h` to label
against, so the **target stays an overpass-to-overpass (~24 h) transition**, just
as in the daily model. What becomes hourly is the **forcing**: temperature,
humidity, vapour pressure deficit and wind are sampled at the exact detection
timestamp and across the preceding 24 h, instead of as a daily mean.

### Why not XHWI

XHWI covers **2024 only**. Extending the model back to 2019 requires a thermal
forcing available in every year, so it comes from raw ERA5 — `t2m`, `rh` and
`vpd`, the physical quantities XHWI is itself built from. The count of hours
above 32 °C reproduces XHWI's temperature threshold as a year-independent
feature. If XHWI ever covers 2019–2024, this notebook detects that on its own and
adds a fourth feature set.

### Feature sets

| set | contents | question it answers |
|---|---|---|
| `baseline` | daily equivalent: daily means/maxima, blocking, SACZ, seasonality, position, size | starting point |
| `with_hourly` | + hourly thermal (`t2m`/`rh`/`vpd` at detection time and lags) and cluster geometry | does hourly forcing help? |
| `with_hourly_wind` | + hourly wind | does **hourly** wind help beyond the **daily** wind already in the baseline? |

`with_hourly` bundles the gain from hourly forcing together with the gain from
cluster geometry — both exist only because of the new data, but this design
cannot attribute between them.

In [ ]:
# System
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# Helpers and Preprocessing
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler

# Metrics for evaluation
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
)

# Optimization
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# NOTE: every create_study below passes an explicitly seeded TPESampler.
# `random_state=42` seeds the ESTIMATOR, not Optuna's search. Without a sampler
# seed the hyperparameter search is nondeterministic, and re-running an
# IDENTICAL configuration moved baseline persistence ROC-AUC from 0.631 to
# 0.601 -- as large as the feature-set deltas being measured. Seeding makes runs
# reproducible; it does not make small deltas significant (see README.md).

np.random.seed(42)

In [ ]:
# this folder, whether the notebook is opened from here or from the repository root
BASE_DIR = Path.cwd()
if not (BASE_DIR / "build_hourly_dataset.py").exists():
    BASE_DIR = BASE_DIR / "hourly_model"
DATA_FILE = BASE_DIR / "data" / "hourly_ml_table_pantanal.parquet"
RESULTS_DIR = BASE_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise SystemExit(
        f"{DATA_FILE.name} not found. Run this first:\n"
        "    python build_hourly_dataset.py"
    )

df = pd.read_parquet(DATA_FILE)
print(f"{len(df):,} clusters | {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"link radius: {df['link_km'].iloc[0]} km | spread ring: up to {df['ring_km'].iloc[0]} km")
df.head(3)

### Dataset overview

Each row is **one cluster of hotspots in one satellite overpass**. The targets
describe what happens at the next overpass:

- `persists_t1` — some cluster is detected within 5 km of this centroid.
- `spreads_t1` — a **new** cluster (one that is not the continuation of an
  already-burning fire) appears in the 5–20 km ring.

Clusters whose next day was not observed regionally were already dropped when the
table was built — absence of detection there is not evidence of extinction.

In [ ]:
print("Target rates:")
print(df[['persists_t1', 'spreads_t1']].mean().round(3).to_string())
print()

yearly = df.groupby('year').agg(
    n_clusters=('event_id', 'size'),
    n_hotspots=('n_hotspots', 'sum'),
    persistence_rate=('persists_t1', 'mean'),
    spread_rate=('spreads_t1', 'mean'),
).round(3)
print("By year:")
print(yearly.to_string())
print()

monthly = df.groupby(df['date'].dt.month).agg(
    n_clusters=('event_id', 'size'),
    n_hotspots=('n_hotspots', 'sum'),
    persistence_rate=('persists_t1', 'mean'),
    spread_rate=('spreads_t1', 'mean'),
).round(3)
monthly.index.name = 'month'
print(monthly.to_string())

print()
print("Detection hour (UTC):")
print(df['detection_hour_utc'].value_counts().sort_index().to_string())
print()
print("Gap to next overpass (h):")
print(df['dt_next_overpass_h'].value_counts().sort_index().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

y = df.groupby('year')
axes[0].bar(y.size().index, y.size().values, color="#9e9e9e")
axes[0].set_xlabel("year"); axes[0].set_ylabel("clusters detected")
axes[0].set_title("Detection volume by year - Pantanal")

m = df.groupby(df['date'].dt.month)

axes[1].plot(m['persists_t1'].mean().index, m['persists_t1'].mean().values,
             marker='o', label='persists_t1', color="#2e7d32")
axes[1].plot(m['spreads_t1'].mean().index, m['spreads_t1'].mean().values,
             marker='s', label='spreads_t1', color="#c62828")
axes[1].set_xlabel("month"); axes[1].set_ylabel("rate")
axes[1].set_title("Target rates by month (all years pooled)")
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "overview_by_year_and_month.png", dpi=200, bbox_inches="tight")
plt.show()

### Feature set definitions

Note that `movement_bearing_deg` and `cluster_axis_deg` are **not** used as
features: they are circular angles, and enter only through their sine/cosine
encodings (`cluster_axis_sin/cos`) or through the derived `wind_alignment`.

`dt_next_overpass_h` is computed from the **regional** overpass time (MS/MT), not
from Pantanal detections. Taken from the Pantanal alone it would be undefined
exactly on the days nothing burned there — and any fill value would then encode
the answer to `persists_t1`.

`timing_features` (`dt_next_overpass_h`, `detection_hour_utc`) are **defined but
excluded**. An ablation (see `README.md`) found they contribute nothing —
removing them *improves* persistence ROC-AUC from 0.636 to 0.648 and leaves
spread unchanged — despite ranking high in XGBoost's importance. Importance
counts how often a feature was split on, not whether those splits paid off.

In [ ]:
baseline_features = [
    # position and size
    'lat', 'lon', 'n_hotspots', 'persistence_accumulated',
    # cyclical seasonality + interannual trend
    'day_year_sin', 'day_year_cos', 'month_sin', 'month_cos', 'year',
    # DAILY forcing -- what the daily model was able to see
    't2m_daily_mean', 't2m_daily_max', 'rh_daily_mean', 'rh_daily_max',
    'vpd_daily_mean', 'vpd_daily_max',
    'wind_speed_daily_mean', 'wind_speed_daily_max',
    # atmospheric blocking
    'total', 'norte', 'norte_h1', 'norte_h2', 'sul', 'sul_h1', 'sul_h2',
    # SACZ
    'SACZ_AB', 'SACZ_C', 'SACZ_DE',
]

hourly_thermal_features = [
    't2m_t', 't2m_lag1', 't2m_lag3', 't2m_lag6', 't2m_lag12',
    't2m_24h_max', 't2m_24h_mean', 't2m_prev12h_min',
    't2m_24h_hours_gt32', 't2m_24h_hours_gt35',
    'rh_t', 'rh_lag3', 'rh_lag6', 'rh_lag12',
    'rh_24h_min', 'rh_24h_mean', 'rh_prev12h_min', 'rh_24h_hours_lt30',
    'vpd_t', 'vpd_lag3', 'vpd_lag6', 'vpd_lag12',
    'vpd_24h_max', 'vpd_24h_mean', 'vpd_prev12h_min', 'vpd_24h_hours_gt20',
]

geometry_features = [
    'cluster_spread_km', 'cluster_radius_km', 'cluster_elongation',
    'has_axis', 'cluster_axis_sin', 'cluster_axis_cos',
    'nn_dist_km', 'has_same_day_neighbor', 'n_clusters_ring',
    'movement_km', 'has_prior_position',
]

# EXCLUDED from the feature sets. These rank high in XGBoost's importance but an
# ablation showed they contribute nothing: removing them *improves* persistence
# (ROC-AUC 0.636 -> 0.648) and leaves spread flat. Importance counts how often a
# feature was split on, not whether those splits helped. Kept defined so the
# decision stays visible and is one edit to reverse.
timing_features = ['dt_next_overpass_h', 'detection_hour_utc']

# Only what HOURLY wind adds: the daily wind means are already in the baseline,
# so this block's delta answers "does hourly wind help beyond daily wind?",
# which is the sharper question.
wind_features = [
    'wind_speed_t', 'wind_speed_lag3', 'wind_speed_lag6', 'wind_speed_lag12',
    'wind_speed_24h_max', 'wind_speed_24h_mean', 'wind_speed_prev12h_min',
    'wind_dir_var_24h', 'wind_u_t', 'wind_v_t',
    'wind_dir_sin', 'wind_dir_cos', 'wind_alignment',
]

xhwi_features = [
    'xhwi_t', 'xhwi_lag1', 'xhwi_lag3', 'xhwi_lag6', 'xhwi_lag12',
    'xhwi_24h_max', 'xhwi_24h_mean', 'xhwi_prev12h_min',
    'xhwi_daily_mean', 'xhwi_daily_max',
    'xhwi_24h_hours_pos', 'xhwi_24h_hours_gt01', 'xhwi_hours_since_peak',
]

hourly_block = hourly_thermal_features + geometry_features

feature_sets = {
    'baseline':          list(baseline_features),
    'with_hourly':       list(baseline_features) + hourly_block,
    'with_hourly_wind':  list(baseline_features) + hourly_block + wind_features,
}

# XHWI only enters if it covers EVERY row. Today it exists for 2024 only, so on a
# 2019-2024 span this block is left out automatically -- rather than silently
# discarding five years of data to accommodate one feature.
have_xhwi = [c for c in xhwi_features if c in df.columns]
if have_xhwi and df[have_xhwi].notna().all(axis=1).all():
    feature_sets['with_hourly_wind_xhwi'] = (
        list(baseline_features) + hourly_block + wind_features + have_xhwi)
    print("XHWI covers every row -- fourth feature set added.\n")
elif have_xhwi:
    cov = df[have_xhwi].notna().all(axis=1)
    years = sorted(df.loc[cov, 'year'].unique())
    print(f"XHWI covers {cov.sum():,}/{len(df):,} rows (years {years}) -- "
          f"block excluded from the feature sets.\n")

targets = ['persists_t1', 'spreads_t1']

# Per-set colours, used by every plot further down.
COLOR_BASE, COLOR_HOURLY, COLOR_WIND, COLOR_XHWI = "#9e9e9e", "#1565c0", "#2e7d32", "#ef6c00"
SET_COLORS = {
    "baseline": COLOR_BASE,
    "with_hourly": COLOR_HOURLY,
    "with_hourly_wind": COLOR_WIND,
    "with_hourly_wind_xhwi": COLOR_XHWI,
}

for name, feats in feature_sets.items():
    print(f"{name:24s} {len(feats):3d} features")

missing = {n: [f for f in fs if f not in df.columns] for n, fs in feature_sets.items()}
missing = {n: m for n, m in missing.items() if m}
print("\nMissing columns:", missing if missing else "none")

for name, feats in feature_sets.items():
    nan_counts = df[feats].isna().sum()
    nan_counts = nan_counts[nan_counts > 0]
    if len(nan_counts):
        print(f"\nWARNING - NaN in [{name}]:")
        print(nan_counts.to_string())
    else:
        print(f"[{name}] no NaN.")

### Train / test split

With 2019–2024 available, the daily notebook's **split by year** becomes possible
again, instead of cutting inside a single year: train **2019–2022**, test
**2023–2024**.

The years are very uneven — 2020 was the Pantanal's catastrophic season (9,851
clusters) and 2022 barely burned (845). Training carries both the extreme and the
quiet years; the test set gets one weak year (2023) and one strong one (2024),
which is a reasonable sample of regimes.

**Chain purge:** a fire spanning 31 Dec 2022 – 1 Jan 2023 would appear on both
sides of the cut. Any training row whose chain (`chain_id`) also has rows in the
test set is removed.

In [ ]:
YEAR_TRAIN_END = 2022  # test starts in 2023

df = df.sort_values("detection_hour").reset_index(drop=True)

test_mask = df["year"] > YEAR_TRAIN_END
test_chains = set(df.loc[test_mask, "chain_id"].unique())

train_mask = (~test_mask) & (~df["chain_id"].isin(test_chains))
purged = (~test_mask).sum() - train_mask.sum()

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"Train: {df_train['date'].min().date()} -> {df_train['date'].max().date()}  "
      f"({len(df_train):,} clusters)")
print(f"Test : {df_test['date'].min().date()} -> {df_test['date'].max().date()}  "
      f"({len(df_test):,} clusters)")
print(f"Rows purged for chains crossing the cut: {purged}")
print()
print("Target rates:")
print(pd.DataFrame({
    "train": df_train[targets].mean(),
    "test": df_test[targets].mean(),
}).round(3).to_string())
print()
print("Clusters per year:")
print(df.groupby("year").size().to_string())

### Training and evaluation functions

Same model families and same strategy as the daily notebook: Optuna over
`TimeSeriesSplit`, `f1_macro` for persistence (a reasonably balanced target) and
`average_precision` with threshold calibration for spread (the minority target).

Since there is a single biome here, the `ExpandingWindowSplit` that existed for
the Pampa is not needed.

In [ ]:
def calculate_scale_pos_weight(y_train):
    '''Compensates for imbalance: neg/pos, per XGBoost documentation.'''
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    return n_neg / n_pos if n_pos > 0 else 1.0


def _xgb_average_precision_scorer(estimator, X_, y_):
    '''Manually slices the positive-class column before scoring.

    Carried over from the daily notebook: some xgboost/scikit-learn pairings
    hand the full (n, 2) predict_proba output to average_precision_score, which
    raises inside cross_val_score and is silently swallowed as a NaN fold score
    -- every trial then scores NaN and Optuna reports 'No trials are completed'
    with no indication why. Slicing the column ourselves sidesteps it.
    '''
    proba = estimator.predict_proba(X_)[:, 1]
    return average_precision_score(y_, proba)


def optimize_xgboost_persist(X, y, n_trials=20, cv=None, scoring="f1_macro", random_state=42):
    def objective(trial):
        model = XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 800),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_weight=trial.suggest_float("min_child_weight", 1.0, 10.0),
            gamma=trial.suggest_float("gamma", 0.0, 5.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", random_state=random_state, n_jobs=-1,
        )
        return np.mean(cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=-1))

    study = optuna.create_study(
        direction="maximize", study_name="XGBoost_persist",
        sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials)

    best_model = XGBClassifier(
        **study.best_params, objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", random_state=random_state, n_jobs=-1,
    )
    best_model.fit(X, y)
    print(f"  XGBoost (persist): best {scoring}={study.best_trial.value:.4f}")
    return "XGBoost", best_model, study.best_value


def optimize_random_forest_persist(X, y, n_trials=20, cv=None, scoring="f1_macro", random_state=42):
    def objective(trial):
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 450),
            max_depth=trial.suggest_int("max_depth", 3, 30),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced_subsample"]),
            random_state=random_state, n_jobs=-1,
        )
        return np.mean(cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=1))

    study = optuna.create_study(
        direction="maximize", study_name="RandomForest_persist",
        sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials)

    best_model = RandomForestClassifier(**study.best_params, random_state=random_state, n_jobs=-1)
    best_model.fit(X, y)
    print(f"  RandomForest (persist): best {scoring}={study.best_trial.value:.4f}")
    return "RandomForest", best_model, study.best_value


def optimize_xgboost_prop(X, y, n_trials=20, cv=None, scoring="average_precision", random_state=42):
    scale_pos_weight = calculate_scale_pos_weight(y)
    cv_scoring = _xgb_average_precision_scorer if scoring == "average_precision" else scoring

    def objective(trial):
        model = XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 800),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_weight=trial.suggest_float("min_child_weight", 1.0, 10.0),
            gamma=trial.suggest_float("gamma", 0.0, 5.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            scale_pos_weight=scale_pos_weight,
            objective="binary:logistic", eval_metric="aucpr",
            tree_method="hist", random_state=random_state, n_jobs=-1,
        )
        return np.mean(cross_val_score(model, X, y, cv=cv, scoring=cv_scoring, n_jobs=-1))

    study = optuna.create_study(
        direction="maximize", study_name="XGBoost_prop",
        sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials)

    best_model = XGBClassifier(
        **study.best_params, scale_pos_weight=scale_pos_weight,
        objective="binary:logistic", eval_metric="aucpr",
        tree_method="hist", random_state=random_state, n_jobs=-1,
    )
    best_model.fit(X, y)
    print(f"  XGBoost (spread): best {scoring}={study.best_trial.value:.4f} | spw={scale_pos_weight:.2f}")
    return "XGBoost", best_model, study.best_value


def optimize_random_forest_prop(X, y, n_trials=20, cv=None, scoring="average_precision", random_state=42):
    def objective(trial):
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 450),
            max_depth=trial.suggest_int("max_depth", 3, 30),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
            class_weight=trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample"]),
            random_state=random_state, n_jobs=-1,
        )
        return np.mean(cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=1))

    study = optuna.create_study(
        direction="maximize", study_name="RandomForest_prop",
        sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials)

    best_model = RandomForestClassifier(**study.best_params, random_state=random_state, n_jobs=-1)
    best_model.fit(X, y)
    print(f"  RandomForest (spread): best {scoring}={study.best_trial.value:.4f}")
    return "RandomForest", best_model, study.best_value

In [ ]:
def get_positive_scores(model, X, positive_class=1):
    y_proba = model.predict_proba(X)
    classes = list(model.classes_)
    return y_proba[:, classes.index(positive_class)]


def compute_binary_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true)
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "n": len(y_true),
        "prevalence_positive": np.mean(y_true == 1),
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_positive": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "precision_positive": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_positive": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }
    if len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
        out["pr_auc"] = average_precision_score(y_true, y_score)
    else:
        out["roc_auc"] = np.nan
        out["pr_auc"] = np.nan
    return out


def find_best_threshold_prop(model, X, y, positive_class=1, metric="f1_positive"):
    y = np.asarray(y)
    classes = list(model.classes_)
    y_score = model.predict_proba(X)[:, classes.index(positive_class)]
    precision, recall, thresholds = precision_recall_curve(y, y_score, pos_label=positive_class)

    precision_t, recall_t = precision[:-1], recall[:-1]
    f1_t = 2 * precision_t * recall_t / (precision_t + recall_t + 1e-12)

    if metric == "f1_positive":
        best_idx = np.argmax(f1_t)
    elif metric == "precision_recall_balance":
        best_idx = np.argmin(np.abs(precision_t - recall_t))
    else:
        raise ValueError("metric must be 'f1_positive' or 'precision_recall_balance'")

    return {
        "best_threshold": thresholds[best_idx],
        "best_precision": precision_t[best_idx],
        "best_recall": recall_t[best_idx],
        "best_f1_positive": f1_t[best_idx],
        "metric": metric,
    }

### Final training — baseline vs. hourly vs. hourly + wind

Runs the same pipeline once per feature set. `N_TRIALS_FINAL` is 20, matching the
daily notebook. For a quick check, drop it to 3–5 before running the whole cell.

Besides metrics on the full test set, the **dry season** (May–Oct) is scored
separately — that is when the Pantanal actually burns, and averaging over the
whole test period dilutes performance with November and December, months of
almost no activity.

In [ ]:
DRY_MONTHS = [5, 6, 7, 8, 9, 10]
N_TRIALS_FINAL = 20

final_metrics = []
final_predictions = []
final_feature_importance = []
final_models = {}

target_info = {
    "persists_t1": {
        "label": "Persistence",
        "model_functions": [
            ("XGBoost", optimize_xgboost_persist),
            ("RandomForest", optimize_random_forest_persist),
        ],
        "scoring": "f1_macro",
        "use_threshold_calibration": False,
    },
    "spreads_t1": {
        "label": "Spread",
        "model_functions": [
            ("XGBoost", optimize_xgboost_prop),
            ("RandomForest", optimize_random_forest_prop),
        ],
        "scoring": "average_precision",
        "use_threshold_calibration": True,
    },
}

for feature_set_name, feature_list in feature_sets.items():

    print("\n" + "#" * 80)
    print(f"FEATURE SET: {feature_set_name} ({len(feature_list)} features)")
    print("#" * 80)

    for target_col, info in target_info.items():

        print(f"\n=== {target_col} - {info['label']} ===")

        if info["use_threshold_calibration"]:
            # Hold out the tail of the training period to calibrate the
            # decision threshold, so it is never tuned on the test set.
            split_idx = int(0.8 * len(df_train))
            df_model_train = df_train.iloc[:split_idx].copy()
            df_threshold_val = df_train.iloc[split_idx:].copy()

            X_train_final = df_model_train[feature_list]
            y_train_final = df_model_train[target_col]
            X_threshold_val = df_threshold_val[feature_list]
            y_threshold_val = df_threshold_val[target_col]
        else:
            X_train_final = df_train[feature_list]
            y_train_final = df_train[target_col]
            X_threshold_val = None
            y_threshold_val = None

        cv = TimeSeriesSplit(n_splits=5)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_final)

        for model_label, opt_function in info["model_functions"]:

            print(f"\nTraining {model_label} | {info['label']} | {feature_set_name}")

            model_name, model, cv_score = opt_function(
                X=X_train_scaled, y=y_train_final, n_trials=N_TRIALS_FINAL,
                cv=cv, scoring=info["scoring"], random_state=42,
            )

            if info["use_threshold_calibration"] and y_threshold_val.nunique() == 2:
                X_threshold_scaled = scaler.transform(X_threshold_val)
                threshold = find_best_threshold_prop(
                    model=model, X=X_threshold_scaled, y=y_threshold_val,
                    positive_class=1, metric="f1_positive",
                )["best_threshold"]
            else:
                threshold = 0.5

            X_test = df_test[feature_list]
            y_test = df_test[target_col]
            X_test_scaled = scaler.transform(X_test)

            y_score = get_positive_scores(model, X_test_scaled, positive_class=1)
            y_pred = (y_score >= threshold).astype(int)

            tag = {
                "target": target_col, "target_label": info["label"], "biome": "Pantanal",
                "model": model_label, "cv_score": cv_score,
                "feature_set": feature_set_name, "n_features": len(feature_list),
            }

            m_all = compute_binary_metrics(y_test, y_score, threshold)
            m_all.update(tag); m_all["period"] = "test_full"
            final_metrics.append(m_all)

            dry_mask = df_test["date"].dt.month.isin(DRY_MONTHS)
            if dry_mask.sum() > 0 and y_test.loc[dry_mask].nunique() == 2:
                m_dry = compute_binary_metrics(
                    y_test.loc[dry_mask], y_score[dry_mask.values], threshold)
                m_dry.update(tag); m_dry["period"] = "test_dry_season"
                final_metrics.append(m_dry)

            pred_df = df_test[["date", "detection_hour", "lat", "lon",
                               "n_hotspots", "persistence_accumulated"]].copy()
            pred_df["target"] = target_col
            pred_df["target_label"] = info["label"]
            pred_df["model"] = model_label
            pred_df["feature_set"] = feature_set_name
            pred_df["y_true"] = y_test.values
            pred_df["y_score"] = y_score
            pred_df["y_pred"] = y_pred
            pred_df["threshold"] = threshold
            pred_df["period"] = np.where(
                pred_df["date"].dt.month.isin(DRY_MONTHS), "dry_season", "wet_season")
            final_predictions.append(pred_df)

            if hasattr(model, "feature_importances_"):
                final_feature_importance.append(pd.DataFrame({
                    "target": target_col, "target_label": info["label"],
                    "model": model_label, "feature_set": feature_set_name,
                    "feature": feature_list, "importance": model.feature_importances_,
                }))

            final_models[(feature_set_name, target_col, model_label)] = {
                "model": model, "scaler": scaler, "threshold": threshold, "cv_score": cv_score,
            }

df_final_metrics = pd.DataFrame(final_metrics)
df_final_predictions = pd.concat(final_predictions, ignore_index=True)
df_final_feature_importance = pd.concat(final_feature_importance, ignore_index=True)

df_final_metrics

In [ ]:
df_final_metrics.to_csv(RESULTS_DIR / "final_test_metrics_hourly.csv", index=False)
df_final_predictions.to_csv(RESULTS_DIR / "final_test_predictions_hourly.csv", index=False)
df_final_feature_importance.to_csv(RESULTS_DIR / "feature_importance_hourly.csv", index=False)
print("saved to", RESULTS_DIR)

### Comparison across the feature sets

This is the point of the design: **measure** whether hourly forcing helps, rather
than assume it. Deltas are computed against `baseline`.

In [ ]:
metric_cols = ["accuracy", "f1_macro", "f1_positive", "precision_positive",
               "recall_positive", "roc_auc", "pr_auc"]

df_compare = df_final_metrics.pivot_table(
    index=["target_label", "model", "period"], columns="feature_set", values=metric_cols,
).swaplevel(axis=1).sort_index(axis=1)

other_sets = [fs for fs in feature_sets if fs != "baseline"]

for fs in other_sets:
    for metric in metric_cols:
        if ("baseline", metric) in df_compare.columns and (fs, metric) in df_compare.columns:
            df_compare[(f"delta_{fs}", metric)] = (
                df_compare[(fs, metric)] - df_compare[("baseline", metric)]
            )

df_compare

In [ ]:
headline = {"persists_t1": "f1_macro", "spreads_t1": "pr_auc"}

summary = []
for target_col, metric in headline.items():
    for period in ["test_full", "test_dry_season"]:
        d = df_final_metrics[
            (df_final_metrics["target"] == target_col)
            & (df_final_metrics["period"] == period)
        ]
        if d.empty:
            continue
        for model in d["model"].unique():
            dm = d[d["model"] == model].set_index("feature_set")[metric]
            if "baseline" not in dm.index:
                continue
            row = {"target": target_col, "metric": metric, "period": period, "model": model}
            for fs in feature_sets:
                if fs in dm.index:
                    row[fs] = round(dm[fs], 4)
            for fs in other_sets:
                if fs in dm.index:
                    row[f"delta_{fs}"] = round(dm[fs] - dm["baseline"], 4)
            summary.append(row)

df_summary = pd.DataFrame(summary)
df_summary.to_csv(RESULTS_DIR / "comparison_summary_hourly.csv", index=False)
print(df_summary.to_string(index=False))

In [ ]:
def plot_feature_set_comparison(target_col, metric, period="test_full"):
    d = df_final_metrics[
        (df_final_metrics["target"] == target_col) & (df_final_metrics["period"] == period)
    ].copy()
    if d.empty:
        print(f"No data for {target_col} / {period}")
        return

    order = [fs for fs in feature_sets if fs in set(d["feature_set"])]
    plt.figure(figsize=(9, 5))
    ax = sns.barplot(data=d, x="model", y=metric, hue="feature_set",
                     hue_order=order, palette=[SET_COLORS[fs] for fs in order])
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)

    prev = d["prevalence_positive"].iloc[0]
    if metric == "pr_auc":
        ax.axhline(prev, ls="--", c="#c62828", lw=1,
                   label=f"prevalence ({prev:.3f})")
        ax.legend()

    plt.title(f"{metric} - {d['target_label'].iloc[0]} | Pantanal 2019-2024 ({period})")
    plt.ylabel(metric); plt.xlabel("")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"comparison_{target_col}_{metric}_{period}.png",
                dpi=300, bbox_inches="tight")
    plt.show()


plot_feature_set_comparison("spreads_t1", "pr_auc")
plot_feature_set_comparison("spreads_t1", "f1_positive")
plot_feature_set_comparison("persists_t1", "f1_macro")
plot_feature_set_comparison("persists_t1", "roc_auc")
plot_feature_set_comparison("spreads_t1", "pr_auc", period="test_dry_season")
plot_feature_set_comparison("persists_t1", "f1_macro", period="test_dry_season")

### Feature importance

Grey = `baseline` features (daily equivalent), blue = hourly block (hourly
thermal and cluster geometry), green = hourly wind.

In [ ]:
def feature_color(f):
    if f in xhwi_features:
        return COLOR_XHWI
    if f in wind_features:
        return COLOR_WIND
    if f in hourly_block:
        return COLOR_HOURLY
    return COLOR_BASE


BEST_SET = list(feature_sets)[-1]  # the most complete set available


def plot_importance(target_col, model_name, feature_set=None, top_n=20):
    feature_set = feature_set or BEST_SET
    d = df_final_feature_importance[
        (df_final_feature_importance["target"] == target_col)
        & (df_final_feature_importance["model"] == model_name)
        & (df_final_feature_importance["feature_set"] == feature_set)
    ].copy()
    if d.empty:
        print(f"No importance for {target_col} / {model_name} / {feature_set}")
        return

    d = d.sort_values("importance", ascending=False).head(top_n).sort_values("importance")
    colors = [feature_color(f) for f in d["feature"]]

    plt.figure(figsize=(8, max(5, 0.32 * len(d))))
    plt.barh(d["feature"], d["importance"], color=colors)
    plt.xlabel("importance")
    plt.title(f"Top {top_n} features - {target_col} | {model_name} | {feature_set}")

    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (COLOR_BASE, COLOR_HOURLY, COLOR_WIND)]
    plt.legend(handles, ["baseline (daily)", "hourly block", "hourly wind"],
               loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"importance_{target_col}_{model_name}_{feature_set}.png",
                dpi=300, bbox_inches="tight")
    plt.show()


plot_importance("spreads_t1", "XGBoost")
plot_importance("spreads_t1", "RandomForest")
plot_importance("persists_t1", "XGBoost")
plot_importance("persists_t1", "RandomForest")

In [ ]:
# How much of the total importance each block carries
blocks = {"baseline (daily)": baseline_features,
          "hourly block": hourly_block,
          "hourly wind": wind_features,
          "XHWI": xhwi_features}

rows = []
for (target_col, model_name), d in df_final_feature_importance[
    df_final_feature_importance["feature_set"] == BEST_SET
].groupby(["target", "model"]):
    total = d["importance"].sum()
    row = {"target": target_col, "model": model_name}
    for block_name, feats in blocks.items():
        row[block_name] = round(d[d["feature"].isin(feats)]["importance"].sum() / total, 3)
    rows.append(row)

df_blocks = pd.DataFrame(rows)
df_blocks.to_csv(RESULTS_DIR / "importance_by_block_hourly.csv", index=False)
print(f"Share of total importance by block (feature set {BEST_SET}):")
print(df_blocks.to_string(index=False))

### Precision–recall curves

For spread, `pr_auc` is only informative against the **prevalence** (the dashed
line) — a random classifier lands exactly there.

In [ ]:
def plot_pr_curves(target_col, model_name="XGBoost"):
    d = df_final_predictions[
        (df_final_predictions["target"] == target_col)
        & (df_final_predictions["model"] == model_name)
    ]
    if d.empty:
        print(f"No predictions for {target_col} / {model_name}")
        return

    plt.figure(figsize=(7, 6))
    colors = SET_COLORS

    for fs, sub in d.groupby("feature_set"):
        precision, recall, _ = precision_recall_curve(sub["y_true"], sub["y_score"])
        ap = average_precision_score(sub["y_true"], sub["y_score"])
        plt.plot(recall, precision, label=f"{fs} (AP={ap:.3f})",
                 color=colors.get(fs), lw=2)

    prev = d["y_true"].mean()
    plt.axhline(prev, ls="--", c="#c62828", lw=1, label=f"prevalence ({prev:.3f})")
    plt.xlabel("recall"); plt.ylabel("precision")
    plt.title(f"Precision-recall curve - {target_col} | {model_name} | test")
    plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"pr_curve_{target_col}_{model_name}.png",
                dpi=300, bbox_inches="tight")
    plt.show()


plot_pr_curves("spreads_t1", "XGBoost")
plot_pr_curves("persists_t1", "XGBoost")

In [ ]:
# Confusion matrices on the most complete feature set
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, target_col in zip(axes, targets):
    d = df_final_predictions[
        (df_final_predictions["target"] == target_col)
        & (df_final_predictions["model"] == "XGBoost")
        & (df_final_predictions["feature_set"] == BEST_SET)
    ]
    if d.empty:
        continue
    cm = confusion_matrix(d["y_true"], d["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["no", "yes"], yticklabels=["no", "yes"])
    ax.set_title(f"{target_col} | XGBoost | {BEST_SET}")
    ax.set_xlabel("predicted"); ax.set_ylabel("observed")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrices_hourly.png", dpi=300, bbox_inches="tight")
plt.show()